In [16]:
import os

import pandas as pd
from playwright.async_api import async_playwright

HEADERS = [
    "No",
    "Nama Kawasan",
    "Lokasi",
    "Luas Lahan (Ha)",
    "Kegiatan Utama",
    "Nomor Kontak",
    "Telepon",
]

BASE = "https://oss.go.id/id/lokasi-usaha?path=kek&tab=kek&p={p}"
OUTPUT_DIR = "/Users/shaanbarca/Desktop/projects/eez/outputs/data"

In [ ]:
async def extract_rows(page):
    # Try classic <table>
    if await page.locator("table tbody tr").count() > 0:
        rows = await page.locator("table tbody tr").all()
        out = []
        for r in rows:
            cells = await r.locator("td").all_inner_texts()
            cells = [c.strip() for c in cells if c.strip()]
            if cells:
                out.append(cells)
        return out

    # Fallback: ARIA grid/table
    row_locs = page.locator("[role='row']")
    out = []
    for i in range(await row_locs.count()):
        row = row_locs.nth(i)
        cells = await row.locator("[role='cell'], [role='gridcell']").all_inner_texts()
        cells = [c.strip() for c in cells if c.strip()]
        # Skip header-like / empty rows
        if cells and len(cells) >= 5:
            out.append(cells)
    return out


def coerce_to_headers(rows):
    """
    Normalize each row to match HEADERS length.
    Handles these common cases:
    - Row includes "No" already (len == 7)
    - Row excludes "No" (len == 6) -> we generate it
    - Row includes extra junk (len > 7) -> keep last 7 (often safest), or trim
    """
    normalized = []
    for idx, r in enumerate(rows, start=1):
        # If row length is 7, assume it already includes "No"
        if len(r) == 7:
            normalized.append(r)

        # If row length is 6, assume it's missing "No" -> prepend index
        elif len(r) == 6:
            normalized.append([str(idx)] + r)

        # If longer than 7, try to find the best 7-cell window
        elif len(r) > 7:
            # common pattern: sometimes there's an extra first column; prefer last 7
            candidate = r[-7:]
            normalized.append(candidate)

        # If shorter than 6, skip (likely not a real data row)
        else:
            continue

    return pd.DataFrame(normalized, columns=HEADERS)


async def scrape_3_pages():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        dfs = []
        for pno in range(1, 4):
            await page.goto(BASE.format(p=pno), wait_until="domcontentloaded")
            await page.wait_for_timeout(4000)

            rows = await extract_rows(page)
            df = coerce_to_headers(rows)
            df["page"] = pno
            dfs.append(df)

        await browser.close()

    return pd.concat(dfs, ignore_index=True)


df_all = await scrape_3_pages()

In [17]:
df_all.to_csv(os.path.join(OUTPUT_DIR, "oss_kek_all_pages.csv"), index=False)

In [69]:
df_all.to_csv(os.path.join(OUTPUT_DIR, "oss_kek_all_pages.csv"), index=False)

In [ ]:
import json

import requests

# Load what you captured
with open("kek_capture.json", "r", encoding="utf-8") as f:
    cap = json.load(f)

# Find the captured request for sez-regions
req = next(r for r in cap["requests"] if "/api/v1/sez-regions?" in r["url"])
api_url = req["url"]

# Minimal headers needed (authorization is the key one)
headers = {
    "authorization": req["headers"]["authorization"],
    "user-agent": req["headers"]["user-agent"],
    "referer": req["headers"]["referer"],
    # cookie is optional sometimes; include if you get 403/401 without it
    "cookie": req["headers"].get("cookie", ""),
    "accept": "application/json, text/plain, */*",
}

r = requests.get(api_url, headers=headers, timeout=30)
r.raise_for_status()
payload = r.json()

items = payload["data"]["items"]  # <-- confirmed by your preview
df = pd.json_normalize(items)

# Keep useful fields (flattened)
keep = [
    "xid",
    "slug",
    "title",
    "address",
    "latitude",
    "longitude",
    "legalBasis",
    "category.id",
    "category.name",
    "status.id",
    "status.name",
    "createdAt",
    "updatedAt",
]
keep = [c for c in keep if c in df.columns]
df = df[keep].copy()

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"]).drop_duplicates(
    subset=["latitude", "longitude", "title"]
)

df.to_csv(OUTPUT_DIR + "/kek_distribution_points.csv", index=False)
df.head(), df.shape

(        xid                                        slug  \
 0  FHE4Z6LV                        industropolis-batang   
 1  K105PK59  batam-tourism-and-international-healthcare   
 2  DM31BTSP                          bumi-serpong-damai   
 3  WQMIDXT7                                    setangga   
 4  R0DHB01G                                tanjung-sauh   
 
                                                title  \
 0                               Industropolis Batang   
 1         Batam Tourism and International Healthcare   
 2  Banten International Education, Technology, an...   
 3                                           Setangga   
 4                                       Tanjung Sauh   
 
                                              address  latitude   longitude  \
 0                       Batang Regency, Central Java -6.930301  109.961702   
 1                           Batam City, Riau Islands  1.137728  103.924579   
 2                 Tangerang Regency, Banten Province -6.

In [68]:
df.to_csv(OUTPUT_DIR + "/kek_distribution_points.csv", index=False)

In [44]:
print(df.head())

        xid                                        slug  \
0  FHE4Z6LV                        industropolis-batang   
1  K105PK59  batam-tourism-and-international-healthcare   
2  DM31BTSP                          bumi-serpong-damai   
3  WQMIDXT7                                    setangga   
4  R0DHB01G                                tanjung-sauh   

                                               title  \
0                               Industropolis Batang   
1         Batam Tourism and International Healthcare   
2  Banten International Education, Technology, an...   
3                                           Setangga   
4                                       Tanjung Sauh   

                                             address  latitude   longitude  \
0                       Batang Regency, Central Java -6.930301  109.961702   
1                           Batam City, Riau Islands  1.137728  103.924579   
2                 Tangerang Regency, Banten Province -6.308799  106.657530

In [ ]:
req2 = next(r for r in cap["requests"] if "/api/v1/sez-regions/businesses-sector" in r["url"])
r2 = requests.get(req2["url"], headers=headers, timeout=30)
r2.raise_for_status()
payload2 = r2.json()
sectors = payload2["data"]["items"]

df_sec = pd.DataFrame(sectors)
df_sec.to_csv(OUTPUT_DIR + "/kek_business_sectors.csv", index=False)
df_sec.head(), df_sec.shape

(        xid                             name   createdAt     updatedAt
 0  1D1ZXY3G  Area Infrastructure Preparation  1706166883           NaN
 1  4RKQNZTZ              Automotive Industry  1706166883           NaN
 2  CSBPAFC4              Automotive Industry  1770354237  1.770354e+09
 3  6U03KXRL              Base Metal Industry  1706166883           NaN
 4  1PORF5O9                 Bauxite Industry  1706166883           NaN,
 (31, 4))

In [66]:
df_sec.to_csv(OUTPUT_DIR + "/kek_business_sectors.csv", index=False)

In [ ]:
df_points = df.copy()

In [46]:
import json

PAGE_URL = "https://kek.go.id/id/investment/distribution"


async def capture_detail_calls_for(title="Setangga"):
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir="kek_profile",
            headless=False,
            viewport={"width": 1400, "height": 800},
            args=["--disable-blink-features=AutomationControlled"],
        )
        page = await context.new_page()

        matches = []

        async def on_response(resp):
            req = resp.request
            if req.resource_type not in ("xhr", "fetch"):
                return
            u = resp.url.lower()
            if "kek.go.id/api/" not in u:
                return
            if "/api/v1/sez-regions?" in u:  # ignore list call
                return

            if resp.status == 200:
                matches.append(resp.url)
                try:
                    data = await resp.json()
                    print("\n=== API ===")
                    print(resp.url)
                    print(json.dumps(data, ensure_ascii=False)[:500], "...\n")
                except:
                    txt = await resp.text()
                    print("\n=== API (text) ===")
                    print(resp.url)
                    print(txt[:300], "...\n")

        page.on("response", on_response)

        await page.goto(PAGE_URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(8000)

        # Locate the label
        label = page.get_by_text(title, exact=True).first
        await label.wait_for(state="visible")

        # 1) Try force-click (ignores intercepting overlays)
        await label.click(force=True)

        await page.wait_for_timeout(800)

        # Click "Lihat Detail" (force too, just in case)
        await page.get_by_role("button", name="Lihat Detail").click(force=True)

        await page.wait_for_timeout(12000)

        # Print unique URLs
        uniq = []
        seen = set()
        for u in matches:
            if u not in seen:
                seen.add(u)
                uniq.append(u)

        print("\n=== UNIQUE API URLS AFTER LIHAT DETAIL ===")
        for u in uniq:
            print(u)

        await context.close()


await capture_detail_calls_for("Setangga")


=== API ===
https://kek.go.id/api/v2/footer
{"success": true, "code": "200", "message": "OK", "data": {"xid": "SXTZMJMQ", "address": "Gedung MNC Tower Lantai 3, Jl. Kebon Sirih No.17 - 19, Kebon Sirih, Kec. Menteng, Kota Jakarta Pusat, Daerah Khusus Ibukota Jakarta 10340.", "description": "<p>.</p>", "phone": "(021) 3912491", "email": "info@kek.go.id", "imageFiles": {"desktop": {"url": "https://object.kek.go.id/ekon-kek-prod-bucket/footer/3GNNycilVwdgu78tMxg5bLybXdhAy6AE0ELqwYUa.png?X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Algorithm=AW ...


=== API ===
https://kek.go.id/api/v1/sez-regions/businesses-sector?skip=0&limit=1000
{"success": true, "code": "200", "message": "OK", "data": {"items": [{"xid": "ENZTSNY8", "name": "Energi", "createdAt": 1706166883, "updatedAt": null}, {"xid": "QKWMPUWW", "name": "Industri Alat Medis", "createdAt": 1770360083, "updatedAt": 1770360089}, {"xid": "1PORF5O9", "name": "Industri Bauksit", "createdAt": 1706166883, "updatedAt": null}, {"xid": "VOFNY4A

In [50]:
import json

import pandas as pd
import requests

# Load auth header you captured earlier
with open("kek_capture.json", "r", encoding="utf-8") as f:
    cap = json.load(f)

AUTH = cap["requests"][0]["headers"]["authorization"]
UA = cap["requests"][0]["headers"]["user-agent"]

HEADERS = {
    "authorization": AUTH,
    "user-agent": UA,
    "accept": "application/json, text/plain, */*",
    "referer": "https://kek.go.id/id/investment/distribution",
}


def get_show(slug: str) -> dict:
    url = f"https://kek.go.id/api/v2/sez-regions/{slug}/show"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()


# df_points = your dataframe from the list endpoint (must include 'slug')
details = []
for slug in df_points["slug"]:
    payload = get_show(slug)
    # payload structure: {"success":true,...,"data":{...}}
    details.append(payload["data"])

df_show = pd.json_normalize(details)
df_show.to_csv("kek_show_details.csv", index=False)

df_show.head(), df_show.shape

(        xid                                        slug  \
 0  FHE4Z6LV                        industropolis-batang   
 1  K105PK59  batam-tourism-and-international-healthcare   
 2  DM31BTSP                          bumi-serpong-damai   
 3  WQMIDXT7                                    setangga   
 4  R0DHB01G                                tanjung-sauh   
 
                                                title  \
 0                               Industropolis Batang   
 1         Batam Tourism and International Healthcare   
 2  Banten International Education, Technology, an...   
 3                                           Setangga   
 4                                       Tanjung Sauh   
 
                                              address  latitude   longitude  \
 0                       Batang Regency, Central Java -6.930301  109.961702   
 1                           Batam City, Riau Islands  1.137728  103.924579   
 2                 Tangerang Regency, Banten Province -6.

In [65]:
df_show.to_csv(OUTPUT_DIR + "/kek_info_and_markers.csv", index=False)

In [58]:
df_show["regionPolygon"]

0     https://object.kek.go.id/ekon-kek-prod-bucket/...
1     https://object.kek.go.id/ekon-kek-prod-bucket/...
2     https://object.kek.go.id/ekon-kek-prod-bucket/...
3     https://object.kek.go.id/ekon-kek-prod-bucket/...
4     https://object.kek.go.id/ekon-kek-prod-bucket/...
5     https://object.kek.go.id/ekon-kek-prod-bucket/...
6     https://object.kek.go.id/ekon-kek-prod-bucket/...
7     https://object.kek.go.id/ekon-kek-prod-bucket/...
8     https://object.kek.go.id/ekon-kek-prod-bucket/...
9     https://object.kek.go.id/ekon-kek-prod-bucket/...
10    https://object.kek.go.id/ekon-kek-prod-bucket/...
11    https://object.kek.go.id/ekon-kek-prod-bucket/...
12    https://object.kek.go.id/ekon-kek-prod-bucket/...
13    https://object.kek.go.id/ekon-kek-prod-bucket/...
14    https://object.kek.go.id/ekon-kek-prod-bucket/...
15    https://object.kek.go.id/ekon-kek-prod-bucket/...
16    https://object.kek.go.id/ekon-kek-prod-bucket/...
17    https://object.kek.go.id/ekon-kek-prod-buc

In [59]:
import os

import requests

os.makedirs("kek_polygons_kml", exist_ok=True)

for _, r in df_show[["slug", "regionPolygon"]].dropna().iterrows():
    slug = r["slug"]
    url = r["regionPolygon"]
    out = os.path.join("kek_polygons_kml", f"{slug}.kml")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(out, "wb") as f:
        f.write(resp.content)

print("Downloaded KMLs -> kek_polygons_kml/")

Downloaded KMLs -> kek_polygons_kml/


In [63]:
import geopandas as gpd
import pandas as pd

rows = []
for _, r in df_show[["slug", "title"]].iterrows():
    slug = r["slug"]
    path = f"kek_polygons_kml/{slug}.kml"
    if not os.path.exists(path):
        continue
    try:
        g = gpd.read_file(path)
        g["slug"] = slug
        g["title"] = r.get("title")
        rows.append(g)
    except Exception as e:
        print("Failed:", slug, e)

gdf = pd.concat(rows, ignore_index=True)
gdf.to_file(OUTPUT_DIR + "/kek_polygons.geojson", driver="GeoJSON")
gdf[["slug", "title", "geometry"]].head()

,slug,title,geometry
0,industropolis-batang,Industropolis Batang,"MULTIPOLYGON Z (((109.92736 -6.94108 0, 109.92..."
1,batam-tourism-and-international-healthcare,Batam Tourism and International Healthcare,"MULTIPOLYGON Z (((103.93513 1.1245 0, 103.9343..."
2,bumi-serpong-damai,"Banten International Education, Technology, an...","MULTIPOLYGON Z (((106.61279 -6.31251 0, 106.61..."
3,setangga,Setangga,"MULTIPOLYGON (((116.08232 -3.27168, 116.08297 ..."
4,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.16259 1.04792, 104.16242 1..."


,slug,title,geometry
0,industropolis-batang,Industropolis Batang,"MULTIPOLYGON Z (((109.92736 -6.94108 0, 109.92..."
1,batam-tourism-and-international-healthcare,Batam Tourism and International Healthcare,"MULTIPOLYGON Z (((103.93513 1.1245 0, 103.9343..."
2,bumi-serpong-damai,"Banten International Education, Technology, an...","MULTIPOLYGON Z (((106.61279 -6.31251 0, 106.61..."
3,setangga,Setangga,"MULTIPOLYGON (((116.08232 -3.27168, 116.08297 ..."
4,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.16259 1.04792, 104.16242 1..."
5,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.18121 1.04371, 104.18127 1..."
6,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.17009 1.06391, 104.17002 1..."
7,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.17911 1.06808, 104.17911 1..."
8,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.17465 1.06365, 104.17582 1..."
9,tanjung-sauh,Tanjung Sauh,"MULTIPOLYGON (((104.1643 1.06551, 104.16584 1...."
